# 🎯 Week 5 Lab — Student Version
## Classification: Can the Data Tell Us What the Clinician Already Knows?

**Course:** Machine Learning for Movement Science  
**Topics:** Logistic regression · Cross-validation · LOSO · ROC curves · Regularization · Feature selection  
**Dataset:** 480 trials (20 subjects × 8 directions × 3 speeds), loaded from Week 4  
**Key question:** If we had only the EMG data, how close could we get to the clinician's diagnosis?

**Dual pipeline theme:** Throughout this lab we compare two feature representations:
- **Raw EMG** — all 6 peak muscle amplitudes (the full signal)
- **PCA scores** — 2 principal components from Week 4 (a compressed summary)

We will discover that what is good for *summarizing* motor patterns (PCA) can be catastrophic for *classifying* clinical groups.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (cross_val_score, cross_val_predict,
                                      StratifiedKFold, LeaveOneGroupOut,
                                      GridSearchCV, learning_curve)
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve, auc,
                              accuracy_score)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
np.random.seed(42)

# Color palette (same as Week 4)
NAVY = '#1B3A5C'; TEAL = '#2E8B8B'; RED = '#C0392B'; BLUE = '#2471A3'
GRAY = '#888888'


Upload data from Week 4

In [ ]:
from google.colab import files
uploaded = files.upload()

---

## 🟢 Part 1: Load the Data from Week 4

In Week 4, we generated EMG data from 20 subjects (10 healthy, 10 impaired) performing center-out reaches. We saved everything — the raw muscle amplitudes, the PCA scores, the labels — into a pickle file.

We **start from exactly where Week 4 left off**. No new data generation here.


In [ ]:
# Solution 1.1: Load Week 4 data
with open('week4_data.pkl', 'rb') as f:
    w4 = pickle.load(f)

X_raw      = w4['X_raw']        # (480, 6) — peak muscle amplitudes
labels     = w4['labels']       # (480,) — 'healthy' or 'impaired'
targets    = w4['targets']      # (480,) — target direction index (0-7)
subjects   = w4['subjects']     # (480,) — subject ID (0-19)
muscle_names = w4['muscle_names']
target_angles = w4['target_angles']
sc_all     = w4['sc_all']       # (480, 2) — PCA scores from shared fit
pca_all    = w4['pca_all']      # fitted PCA object
scaler_all = w4['scaler_all']   # fitted StandardScaler

# Derived labels
group_binary = (labels == 'impaired').astype(int)   # 0=healthy, 1=impaired
healthy_mask = labels == 'healthy'
impaired_mask = labels == 'impaired'

# Cross-validation objects we'll reuse throughout
logo = LeaveOneGroupOut()
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Dataset: {X_raw.shape[0]} trials × {X_raw.shape[1]} muscles")
print(f"  Muscles: {muscle_names}")
print(f"  Healthy: {healthy_mask.sum()}, Impaired: {impaired_mask.sum()}")
print(f"  Subjects: {len(np.unique(subjects))} ({len(np.unique(subjects[healthy_mask]))} healthy, {len(np.unique(subjects[impaired_mask]))} impaired)")
print(f"  Directions: {len(np.unique(targets))}")


---

## 🟢 Part 2: The Question

In Week 4, we produced this figure — all 480 trials in a shared PCA space, colored by direction. We noticed sub-groups within each directional cluster.

Now we reveal what those sub-groups are.


In [ ]:
# Exercise 2.1: Recap — all trials by direction (from Week 4)
### YOUR CODE HERE ###


**The question for this lab:** The clinician already knows who is healthy and who is impaired. But if we *only* had the EMG data — no labels — how close could we get to that diagnosis?

Classification gives us the tool to answer this.


---

## 🟡 Part 3: Drawing a Line — Logistic Regression

The simplest classifier: find the best dividing line between the two groups.

### Exercise 3.1: Fit logistic regression in PCA space (for visualization)

We start with the 2-PC representation so we can *see* the decision boundary. Later we'll discover this is not the best input for classification — but it's invaluable for building intuition.

Build a pipeline: `StandardScaler → PCA(n_components=2) → LogisticRegression`.

**Note on C = 1.0:** Logistic regression has a parameter `C` that controls regularization strength. Smaller C = stronger regularization (weights kept small, less overfitting). Larger C = model fits more aggressively. We use sklearn's default `C = 1.0` for now — Section 6 will search for the best value.


In [ ]:
# Exercise 3.1: Build and fit the PCA pipeline (for visualization)
### YOUR CODE HERE ###


### Exercise 3.2: Visualize the decision boundary
Plot the classifier's confidence (probability of 'impaired') across PCA space, with the actual data points overlaid. This is why we started with PCA — in 6D muscle space, we couldn't draw this plot.


In [ ]:
# Exercise 3.2: Decision boundary visualization
### YOUR CODE HERE ###


### Exercise 3.3: Now build the raw EMG pipeline

The PCA pipeline is useful for visualization, but does compressing 6 muscles to 2 PCs lose information the classifier needs? Build a second pipeline that uses all 6 raw muscle amplitudes.


In [ ]:
# Exercise 3.3: Raw EMG pipeline (no PCA)
### YOUR CODE HERE ###


---

## 🟡 Part 4: Testing Honestly — Cross-Validation

Training accuracy is meaningless — it's like giving a student the answers during the exam.

### Exercise 4.1: Dual-pipeline comparison — k-fold vs LOSO

Compare **both** pipelines (raw EMG vs PCA) under **both** CV schemes (5-fold vs LOSO) for the binary clinical task. This produces 4 numbers that tell a rich story.

- **5-fold CV** splits trials randomly. Same subject can appear in train AND test → data leakage.
- **LOSO** holds out all trials from one subject. The clinically honest test: can we classify a *new patient*?


In [ ]:
# Exercise 4.1: Dual pipeline x dual CV for binary task
# Compute all 4 combinations: (raw, PCA) x (5-fold, LOSO)
### YOUR CODE HERE ###


### Exercise 4.2: Same dual comparison for 8-direction decoding
Now test direction decoding with both pipelines. Is the raw-vs-PCA gap as large?

**Think first:** In the lecture, we learned that directional information is well-captured by the first 2 PCs. So what do you predict?


In [ ]:
# Exercise 4.2: Dual pipeline for 8-direction decoding
### YOUR CODE HERE ###


### Exercise 4.3: Plot per-subject LOSO accuracy
For the binary task using **raw EMG**, which subjects are easy/hard to classify?


In [ ]:
# Exercise 4.3: Per-subject LOSO accuracy (raw EMG)
### YOUR CODE HERE ###


---

## 🟡 Part 5: Reading the Scoreboard

Accuracy tells us *how often* we're right. Now we ask *where* we go wrong.

### Exercise 5.1: Confusion matrix for 8-direction decoding

Use `cross_val_predict` with LOSO and **raw EMG** (all 480 trials) to get predictions, then build the confusion matrix.

**Motor control context:** If neighboring directions (e.g., 0° and 45°) are confused, that's biomechanically sensible — they share similar muscle patterns. Random confusion would be worrying.


In [ ]:
# Exercise 5.1: Confusion matrix — 8 directions (raw EMG, all 480 trials)
### YOUR CODE HERE ###


### Exercise 5.2: From hard labels to soft probabilities — the threshold concept

The confusion matrix used hard predictions. But logistic regression outputs a **probability**, and to get a label we applied a threshold of P >= 0.5 without saying so. There's nothing sacred about 0.5.

Plot the distribution of predicted probabilities for healthy and impaired trials, then show what happens at three different thresholds.


In [ ]:
# Exercise 5.2: Threshold distributions
### YOUR CODE HERE ###


### Exercise 5.3: ROC curves — dual pipeline comparison

Now construct the ROC curve. Each threshold gives one (FPR, TPR) point; the ROC connects them all. Compare raw EMG vs PCA.


In [ ]:
# Exercise 5.3: Dual ROC curves — raw EMG vs PCA (C=1.0)
### YOUR CODE HERE ###


### 🤔 Thought Exercise: What does AUC < 0.5 mean?

The PCA classifier has AUC = 0.36 — *worse than chance*. How is this possible? Would adding more PCs fix it?


In [ ]:
# Exercise: AUC vs number of PCA components
### YOUR CODE HERE ###


---

## 🟡 Part 6: Can We Do Better?

So far, every pipeline used choices we made by hand: 2 PCA components (Week 4) and C = 1.0 (sklearn's default, introduced in Part 3). These are **hyperparameters**.

### Exercise 6.1: GridSearchCV — find the best n_components and C

The **objective function** is classification accuracy: for each (n_components, C) combination, GridSearchCV runs 5-fold CV and keeps the winner. We use the 8-direction task on healthy subjects.


In [ ]:
# Exercise 6.1: GridSearchCV
### YOUR CODE HERE ###


### Exercise 6.2: Which muscles matter most for diagnosis?

**The clinical question:** Which muscles are driving the diagnosis? If the signal is in shoulder muscles, that points to proximal motor control deficits and guides where to focus rehabilitation.

**The ML tool:** L1 regularization forces unimportant weights to exactly zero. The **order muscles appear** as C increases reveals their diagnostic importance.


In [ ]:
# Exercise 6.2: Regularization paths — which muscles matter?
### YOUR CODE HERE ###


---

## 🟡 Part 7: Does the Feature Representation Matter?

In Week 4, we invested significant effort extracting motor synergies. Synergies are compact and physiologically meaningful. But compact ≠ good for classification. Let's test directly.

### Exercise 7.1: Compare raw EMG vs PCA vs Varimax


In [ ]:
# Exercise 7.1: Feature representation comparison
### YOUR CODE HERE ###


### Exercise 7.2: Learning curve — how much data do we need?

Clinical motor control studies face a constant tension: impaired subjects are hard to recruit. The **learning curve** shows how accuracy changes with training set size.

**Important:** This uses **5-fold CV** (not LOSO) and the x-axis shows training **trials** (divide by 24 for approximate subjects). We use 5-fold because learning curves require varying the training size, which is awkward with LOSO. The **shape** of the curve matters more than absolute values (which are optimistic due to trial-level splitting).


In [ ]:
# Exercise 7.2: Learning curve (raw EMG, 5-fold CV)
### YOUR CODE HERE ###


---

## 🔴 Part 8: The Payoff — How Close Did We Get to the Clinician?

Our best logistic regression: **raw EMG** (all 6 muscles), **C = 10** (tuned), tested with **LOSO**.

### Exercise 8.1: Build the best model and compare to the clinician


In [ ]:
# Exercise 8.1: The payoff — best model vs the clinician
### YOUR CODE HERE ###


In [ ]:
# Exercise 8.2: The payoff figure
# NOTE: Points PLOTTED in PCA space for visualization, but
# the classifier used all 6 raw muscle amplitudes as input.

### YOUR CODE HERE ###


---

## Summary

| What we did | What it told us |
|:---|:---|
| Logistic regression on PCA (Part 3) | Visualize the decision boundary — useful for intuition |
| Raw EMG pipeline (Part 3) | All 6 muscles contribute; can't visualize but classifies better |
| Dual-pipeline CV comparison (Part 4) | Raw EMG (77% LOSO) >> PCA (44%) for clinical diagnosis |
| 8-direction decoding (Part 4) | Both representations work similarly (~81%) — PCA captures directional info |
| Threshold distributions (Part 5) | The 0.5 threshold is a choice; different costs → different thresholds |
| Dual ROC curves (Part 5) | Raw AUC=0.92 (C=1), PCA AUC=0.36 — PCA is worse than chance |
| AUC vs n_components (Part 5) | Diagnostic signal concentrated in PC4; variance ≠ relevance |
| GridSearchCV (Part 6) | Systematic hyperparameter search; direction task easy with 4+ PCs |
| L1 regularization paths (Part 6) | Proximal extensors carry strongest diagnostic signal |
| Feature comparison (Part 7) | Raw EMG >> PCA = Varimax; rotation can't rescue lost signal |
| Learning curve (Part 7) | ~10 subjects gets most of the way; curve not yet plateaued |
| The payoff (Part 8) | Best model: raw EMG, C=10 → **85% accuracy, AUC=0.94** |

**The answer:** Raw EMG with tuned logistic regression gets us 85% of the way to the clinician's diagnosis (AUC = 0.94). PCA compression destroys the diagnostic signal. The remaining errors are the clinically ambiguous borderline cases.

**Next week:** Can more powerful classifiers close the gap?
